In [1]:
from local_pool_price import *
import plotly.express as px
import plotly.io

plotly.io.templates.default = None


def fetch_pool_local_token_prices() -> pd.DataFrame:

    safe_price_calls = build_safe_price_calls()
    backing_calls = build_backing_calls()

    token_local_price = build_all_local_pool_prices()

    token_local_price_call_list = [t.calls for t in token_local_price]
    token_local_price_calls = [c for calls in token_local_price_call_list for c in calls]

    blocks = build_blocks_to_use(ETH_CHAIN)
    df = get_raw_state_by_blocks([*safe_price_calls, *backing_calls, *token_local_price_calls], blocks, ETH_CHAIN)
    pool_spot_price_df, pool_spot_price_discount_to_backing_df, pool_spot_price_spread_with_safe_price_df = (
        _extract_pool_local_spot_price_df(df, token_local_price)
    )
    return df, pool_spot_price_df, pool_spot_price_discount_to_backing_df, pool_spot_price_spread_with_safe_price_df


def _extract_pool_local_spot_price_df(df: pd.DataFrame, token_local_price: list[TokenLocalPoolPriceDetails]):
    pool_spot_price_df = pd.DataFrame(index=df.index)
    pool_spot_price_discount_to_backing_df = pd.DataFrame(index=df.index)
    pool_spot_price_spread_with_safe_price_df = pd.DataFrame(index=df.index)

    for t in token_local_price:
        cols = [c for c in df.columns if t.pool_name in c]

        for spot_price_column in cols:
            first_token = spot_price_column.split("_")[-3]
            second_token = spot_price_column.split("_")[-1]

            pool_spot_price_df[f"{t.pool_name}__{first_token}_spot_price"] = (
                df[spot_price_column] * df[f"{second_token}_safe_price"]
            )
            spot_price = pool_spot_price_df[f"{t.pool_name}__{first_token}_spot_price"]
            backing = df[f"{first_token}_backing"]
            safe_price = df[f"{first_token}_safe_price"]

            pool_spot_price_discount_to_backing_df[f"{t.pool_name}__{first_token}_spot_price_discount_to_backing"] = (
                100 * ((spot_price - backing) / backing)
            )

            pool_spot_price_spread_with_safe_price_df[f"{t.pool_name}__{first_token}_safe_spot_spread"] = 100 * (
                (safe_price - spot_price) / safe_price
            )
    return pool_spot_price_df, pool_spot_price_discount_to_backing_df, pool_spot_price_spread_with_safe_price_df


df, pool_spot_price_df, pool_spot_price_discount_to_backing_df, pool_spot_price_spread_with_safe_price_df = (
    fetch_pool_local_token_prices()
)
px.line(pool_spot_price_discount_to_backing_df, title="Spot Price Discount to Backing")

In [2]:
px.line(pool_spot_price_spread_with_safe_price_df, title="Pool Safe Spot Spread")

In [3]:
px.line(pool_spot_price_df, title="Spot Price Pool")